# 10. Validación Clínica IPS

**Objetivo:** consolidar material clínicamente interpretable para discusión con IPS y reutilización en tesis/documentación.
**Entradas:** artefactos vigentes de `dev`, outputs de preprocesamiento, cierres de modelo, predicciones comparables y error analysis alineado.
**Salidas:** carpeta `data/outputs/material_validacion_ips_<timestamp>/` con resúmenes de preprocesamiento, balance, patrones por clase, comparación entre modelos, errores curados y preguntas para IPS.
**Notebook anterior:** `notebooks/analysis/09_analisis_errores_hibrido.ipynb`.
**Notebook siguiente:** ninguno (material para validación clínica y redacción).

> **Alcance actual:** tarea binaria `ansiedad` vs `depresion`; no hay grupo de control ni clase explícita de comorbilidad en esta fase; `test` y xAI siguen pendientes.


## Técnicas, herramientas y librerías de esta etapa

- **Técnica principal:** consolidación clínica-documental de artefactos ya cerrados en `dev`.
- **Herramientas/librerías:** el notebook usa como backend `scripts/export/generar_material_validacion_ips.py`; la curación posterior vive en `scripts/export/curar_dossier_ips.py`; el cierre documental final vive en `scripts/export/cerrar_fase_ips.py`. Para visualización usa `pandas` e `IPython.display`.
- **Por qué es adecuada aquí:** la generación de paquetes y exportables repetibles es más robusta en script que en celdas manuales. El notebook queda como capa legible para IPS, tutoría y tesis.
- **Limitación:** si se quisiera hacer toda la curación solo dentro del notebook, la reproducibilidad caería y sería más fácil dejar outputs desalineados.
- **Alternativa menos recomendable para esta fase:** notebook monolítico con toda la lógica de exportación; más cómodo para exploración, peor para trazabilidad.


## Resumen ejecutivo del problema

Este notebook traduce el estado actual del pipeline en un paquete útil para psiquiatras y para escritura metodológica.
No cambia la tarea experimental, no reabre la selección del modelo y no usa `test`.

Su propósito es responder de forma explícita:

1. cómo se preprocesó y filtró el corpus;
2. con cuántas notas/pacientes se está modelando realmente;
3. cómo quedó el balance/desbalance y cómo se trató;
4. qué parece captar cada enfoque (`TF-IDF`, `ROBERTA_CLINICAL`, híbrido final);
5. qué errores conviene revisar con IPS;
6. qué vacíos siguen dependiendo de validación experta.


In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / 'data').exists() and (cand / 'scripts').exists():
            return cand
    raise RuntimeError('No se pudo resolver la raíz del repositorio.')

ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / 'scripts' / 'export'))

from generar_material_validacion_ips import generar_material  # noqa: E402

RUN_TAG = os.getenv('IPS_OUTPUT_TAG', '').strip() or pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
OUT_DIR = ROOT / 'data' / 'outputs' / f'material_validacion_ips_{RUN_TAG}'
manifest = generar_material(OUT_DIR, verbose=False)
manifest


{'output_dir': '/Users/manuelnunez/Projects/psych-phenotyping-paraguay/data/outputs/material_validacion_ips_20260317_204307',
 'cierre_dir': '/Users/manuelnunez/Projects/psych-phenotyping-paraguay/data/outputs/cierre_modelos_dev_20260311_102929',
 'error_dir': '/Users/manuelnunez/Projects/psych-phenotyping-paraguay/data/outputs/error_analysis_20260311_102942',
 'train_run_referencia': 'train_C_B_A_llm0_sent0_beto1_tpl0_py_XGB_sin_feat_sin_medication_py_XGB_seed42_20260310_202656',
 'feature_run_referencia': 'fe_20260310_082139_py',
 'transformer_standalone': 'ROBERTA_CLINICAL',
 'backbone_hibrido': 'beto',
 'modelo_hibrido_final': 'B_A_llm0_sent0_beto1_tpl0_py_XGB_sin_feat_sin_medication|py|XGB',
 'test_estado': 'pendiente',
 'xai_estado': 'pendiente'}

## 1. Resumen del preprocesamiento y filtrado

## 1b. Qué entendemos por señal clínica útil

Esta sección no redefine el pipeline. Solo vuelve explícita, para lectura clínica y metodológica, la política operativa centralizada en `utils_shared.keep_entity` y reutilizada por el cierre IPS final.

In [2]:
latest_ips_ptr = ROOT / 'data' / 'outputs' / 'ips_cierre_final_latest.json'
ips_cierre_dir = None
if latest_ips_ptr.exists():
    latest_ips_payload = json.loads(latest_ips_ptr.read_text(encoding='utf-8'))
    ips_cierre_dir = Path(latest_ips_payload['output_dir'])
else:
    candidatos_ips = sorted((ROOT / 'data' / 'outputs').glob('ips_cierre_final_*'))
    ips_cierre_dir = candidatos_ips[-1] if candidatos_ips else None

if ips_cierre_dir is None or not ips_cierre_dir.exists():
    raise FileNotFoundError('No se encontró un paquete `ips_cierre_final_*` para mostrar la definición de señal clínica útil.')

display(Markdown((ips_cierre_dir / 'senal_clinica_util.md').read_text(encoding='utf-8')))

# Qué se considera señal clínica útil

## Fuente operativa
- Pipeline clínico construido con el fork `Spanish_Psych_Phenotyping_PY`, montado sobre `spaCy` y componentes de `medSpaCy`.
- Las entidades candidatas provienen del matcher de reglas clínicas del pipeline (`TargetMatcher` dentro del fork).
- Los atributos de contexto (`is_historical`, `is_hypothetical`, `is_family`, `is_negated`) provienen de la capa de contexto clínico de `medSpaCy`.
- Definición centralizada en `utils_shared.keep_entity`.
- Resumen del docstring vigente: Decide si una entidad debe contarse como evidencia clínica válida. Retorna: (keep, is_patient_neg): - keep: si la entidad se conserva como señal para has_clinical_signal / rule features - is_patient_neg: True si la entidad está negada y la fuente es el paciente (sirve para feature niega_*) Política (MCP): 1) Ruido...

## Regla práctica usada por el pipeline
Una mención se considera **señal clínica útil** si pasa el filtro de aseveración clínica de `keep_entity`.

### Se conserva como señal útil cuando
- la mención no está en contexto histórico, hipotético ni familiar;
- la mención no está negada;
- o está negada, pero la negación es atribuible al paciente.

### No se conserva como señal útil cuando
- la mención pertenece a antecedentes, hipótesis clínicas o historia familiar;
- la negación proviene del médico, de una plantilla o de una fórmula administrativa tipo `sin síntomas`.

## Traducción metodológica
- `has_clinical_signal = 1` significa que la nota contiene al menos una entidad que pasa ese filtro.
- `has_clinical_signal = 0` significa que, aun si el texto contiene palabras clínicas, no quedó evidencia válida bajo esa política.
- La negación del paciente se conserva porque puede aportar fenomenología relevante, por ejemplo estado subjetivo, defensas o insight.
- La negación de plantilla o del médico se descarta porque suele diluir la señal diagnóstica útil.

## Alcance
- Esto es una decisión de limpieza y normalización del EHR, no una decisión diagnóstica final.
- Su objetivo es evitar que el modelo aprenda ruido documental como si fuera evidencia clínica.


In [3]:
display(pd.read_csv(OUT_DIR / 'material_ips_preprocesamiento_resumen.csv'))
display(Markdown((OUT_DIR / 'material_ips_preprocesamiento_resumen.md').read_text(encoding='utf-8')))


,etapa,n_registros,n_pacientes,delta_registros,observacion
0,original_crudo,3155,90,0,Carga inicial previa a deduplicación y limpiez...
1,post_deduplicacion_base,3143,90,-12,Se eliminaron duplicados completos; la plantil...
2,notas_con_plantilla_administrativa,2521,89,0,No son notas eliminadas por sí mismas; la plan...
3,eliminadas_sin_senal_clinica_util,1308,90,-1308,Descarte posterior a denoising y lógica de ase...
4,dataset_final_post_filtrado,1835,90,-1320,Universo final modelado en train/dev/test deno...


# Resumen de preprocesamiento para IPS

## Conteos principales

| etapa                              |   n_registros |   n_pacientes |   delta_registros | observacion                                                                                             |
|:-----------------------------------|--------------:|--------------:|------------------:|:--------------------------------------------------------------------------------------------------------|
| original_crudo                     |          3155 |            90 |                 0 | Carga inicial previa a deduplicación y limpieza de plantilla.                                           |
| post_deduplicacion_base            |          3143 |            90 |               -12 | Se eliminaron duplicados completos; la plantilla administrativa se depuró dentro de la nota.            |
| notas_con_plantilla_administrativa |          2521 |            89 |                 0 | No son notas eliminadas por sí mismas; la plantilla se limpia y luego se evalúa si queda señal clínica. |
| eliminadas_sin_senal_clinica_util  |          1308 |            90 |             -1308 | Descarte posterior a denoising y lógica de aseveración clínica.                                         |
| dataset_final_post_filtrado        |          1835 |            90 |             -1320 | Universo final modelado en train/dev/test denoised.                                                     |

## Distribución por clase
- Antes del filtrado (crudo): {'depresion': 2230, 'ansiedad': 925}
- Después del filtrado clínico: {'depresion': 1279, 'ansiedad': 556}

## Aclaración metodológica
- La plantilla administrativa no dispara por sí sola la eliminación de la nota.
- Primero se limpia el bloque plantilla dentro del texto.
- El descarte ocurre cuando, después de la limpieza y del denoising con reglas de aseveración, no queda señal clínica útil.

## Ejemplos
### Texto crudo
Reposicion de medicacion 2) EXAMEN FISICO GRAL. Y GINECOLOGICO PESO ( ) TALLA ( ) PRESION ARTERIAL ( ) PULSO ( ) TEMPERATURA ( ) 3) METODOS AUXILIARES ----- 4) DIAGNOSTICO PRESUNTIVO ----- 5) TRATAMIENTO CLINICO-OTROS ----- 6) PESO ----- 7) PULSO ----- 8) OBSERVACION ----- / ROTELA DE KERMENGUY, ARIANA Carimbo / Assinatura Emissão 14/06/2025 09:54 Ficha 31 1 Página AMBULATORIO 10947615 Pront.: RODRIGUEZ DE BRITEZ(E10/E11), EUGENIA BEATRIZ Paciente: 402320

### Texto limpiado
Reposicion de medicacion

### Texto eliminado y motivo
Reposicion de medicacion

- Motivo de exclusión sugerido: consulta de reposición/seguimiento sin señal clínica suficiente tras depuración.


## 2. Tamaño final del dataset y balance/desbalance

## 2b. Contraste metodológico: baseline crudo vs filtrado

Este contraste no reemplaza las líneas base oficiales. Se usa para explicar por qué el denoising fue necesario y por qué el problema no debe describirse como si el universo crudo y el universo filtrado fueran equivalentes.

In [4]:
display(Markdown((ips_cierre_dir / 'baseline_crudo_vs_filtrado.md').read_text(encoding='utf-8')))

# Baseline crudo vs filtrado

## Objetivo
Este contraste no reemplaza la línea base oficial de la tesis. Su función es mostrar qué pasa cuando se entrena y evalúa una configuración textual simple sobre universos con distinto nivel de ruido.

## Tabla comparativa
| configuracion               | train_universo   | eval_universo   |   macro_f1 |   balanced_accuracy |   f1_ansiedad |   f1_depresion |   n_eval |   n_train |
|:----------------------------|:-----------------|:----------------|-----------:|--------------------:|--------------:|---------------:|---------:|----------:|
| train_base→dev_base         | train_base       | dev_base        |   0.682764 |            0.688849 |      0.575064 |       0.790464 |      595 |      1911 |
| train_base→dev_denoised     | train_base       | dev_denoised    |   0.763774 |            0.800658 |      0.694215 |       0.833333 |      343 |      1911 |
| train_denoised→dev_base     | train_denoised   | dev_base        |   0.673228 |            0.668579 |      0.539773 |       0.806683 |      595 |      1107 |
| train_denoised→dev_denoised | train_denoised   | dev_denoised    |   0.754539 |            0.784774 |      0.677966 |       0.831111 |      343 |      1107 |

## Lectura metodológica
- `train_base→dev_base` muestra el costo de trabajar sobre un universo todavía lleno de seguimiento, reposición y ruido administrativo.
- `train_base→dev_denoised` sirve como control de qué pasa cuando el modelo ve más volumen de texto pero se lo evalúa en un universo clínicamente más coherente.
- `train_denoised→dev_denoised` es el contraste filtrado equivalente con la misma configuración simple.
- Si el crudo no se desploma sobre `dev_denoised`, eso no invalida el denoising: lo que muestra es que el problema principal no es solo rendimiento, sino coherencia metodológica del universo evaluado.

## Conclusión práctica
- El filtrado sigue siendo necesario para que la tarea diferencial tenga sentido clínico.
- El contraste crudo ayuda a mostrar a la tutora que limpiar el dataset no fue una operación cosmética, sino una forma de separar notas diagnósticas de notas de trámite o seguimiento.


In [5]:
display(pd.read_csv(OUT_DIR / 'material_ips_balance_dataset.csv'))
display(Markdown((OUT_DIR / 'material_ips_balance_dataset.md').read_text(encoding='utf-8')))


,split,n_total,ansiedad_n,depresion_n,ansiedad_prop,depresion_prop,clasificacion_balance
0,global_base,3143,925,2218,0.2943,0.7057,claramente_desbalanceado
1,global_post_filtrado,1835,556,1279,0.3030,0.6970,claramente_desbalanceado
2,train,1107,358,749,0.3234,0.6766,claramente_desbalanceado
3,dev,343,100,243,0.2915,0.7085,claramente_desbalanceado
4,test,385,98,287,0.2545,0.7455,claramente_desbalanceado


# Balance y desbalance del dataset

| split                |   n_total |   ansiedad_n |   depresion_n |   ansiedad_prop |   depresion_prop | clasificacion_balance    |
|:---------------------|----------:|-------------:|--------------:|----------------:|-----------------:|:-------------------------|
| global_base          |      3143 |          925 |          2218 |          0.2943 |           0.7057 | claramente_desbalanceado |
| global_post_filtrado |      1835 |          556 |          1279 |          0.303  |           0.697  | claramente_desbalanceado |
| train                |      1107 |          358 |           749 |          0.3234 |           0.6766 | claramente_desbalanceado |
| dev                  |       343 |          100 |           243 |          0.2915 |           0.7085 | claramente_desbalanceado |
| test                 |       385 |           98 |           287 |          0.2545 |           0.7455 | claramente_desbalanceado |

## Lectura metodológica
- En el corte actual el problema queda `claramente_desbalanceado` en el dataset final modelado.
- La clase mayoritaria es `depresion`; la menor, `ansiedad`.
- Por esta razón se priorizan `macro_f1`, `balanced_accuracy` y F1 por clase en lugar de depender solo de accuracy.

## Cómo se trató el desbalance
- `TF-IDF` usa `class_weight='balanced'` en `LinearSVC`.
- `RandomForest` usa `class_weight='balanced'`.
- `XGBoost` no muestra reponderación explícita de clases ni sampling adicional en el notebook vigente.
- No se detectó resampling/oversampling en el pipeline de desarrollo.
- Los transformers baseline se comparan sobre el mismo `dev_denoised`; no se documenta sampling explícito en 04c.


## 3. Qué se hizo y qué no se hizo todavía

In [6]:
display(Markdown((OUT_DIR / 'material_ips_hecho_vs_pendiente.md').read_text(encoding='utf-8')))


# Qué se hizo y qué no se hizo todavía

## Se hizo
- limpieza de duplicados
- filtrado de plantillas administrativas dentro de la nota
- remoción de notas sin señal clínica útil
- split por paciente
- comparación entre baselines e híbridos
- selección del mejor transformer standalone (`roberta_clinical`)
- selección del backbone del híbrido (`beto`)
- cierre formal del mejor modelo en `dev`
- análisis de errores alineado al modelo final

## No se hizo todavía
- validación clínica final con IPS
- reetiquetado experto consulta por consulta
- incorporación de grupo de control
- evaluación final en `test`
- xAI final


## 4. Patrones finales por clase

In [7]:
display(Markdown((OUT_DIR / 'material_ips_patrones_ansiedad.md').read_text(encoding='utf-8')))
display(Markdown((OUT_DIR / 'material_ips_patrones_depresion.md').read_text(encoding='utf-8')))


# Patrones asociados a ansiedad

Resumen por modelo sobre verdaderos positivos en `dev` para `ansiedad`.

## TF-IDF
- Términos recurrentes: refiere, bien, acude, control, crisis, encuentra, ansiedad, niega, animo, paciente

## ROBERTA_CLINICAL
- Términos recurrentes: refiere, bien, ansiedad, encuentra, control, crisis, acude, animo, niega, estable

## HIBRIDO_FINAL
- Términos recurrentes: refiere, bien, encuentra, animo, niega, estable, duerme, control, ansiedad, acude
- Señales clínicas/reglas activas: Autolesion, Ansiedad, Retraimiento social / aislamiento, Sueno / insomnio, Sintomas ansiosos generales, Animo deprimido, Ideas de muerte, Angustia Miedo Temor, Sintomas somaticos, Abulia


# Patrones asociados a depresion

Resumen por modelo sobre verdaderos positivos en `dev` para `depresion`.

## TF-IDF
- Términos recurrentes: control, paciente, acude, refiere, 10, f31, noche, siente, pcte, aea

## ROBERTA_CLINICAL
- Términos recurrentes: control, paciente, acude, refiere, 10, noche, f31, siente, pcte, hace

## HIBRIDO_FINAL
- Términos recurrentes: control, paciente, refiere, acude, 10, noche, f31, siente, pcte, hace
- Señales clínicas/reglas activas: Ideacinsuicida, Autolesion, Ansiedad, Ideas de muerte, Angustia Miedo Temor, Animo deprimido, Retraimiento social / aislamiento, Llantofcil, Sintomas somaticos, Sueno / insomnio


## 5. Qué identifica cada modelo y cuánto se parecen

In [8]:
display(pd.read_csv(OUT_DIR / 'comparacion_aportes_modelos.csv'))
display(Markdown((OUT_DIR / 'comparacion_aportes_modelos.md').read_text(encoding='utf-8')))


,tipo,modelo_a,modelo_b,clase,n_casos,detalle
0,correctos_total,TF-IDF,NaN,todas,263,NaN
1,correctos_por_clase,TF-IDF,NaN,ansiedad,77,NaN
2,correctos_por_clase,TF-IDF,NaN,depresion,186,NaN
3,correctos_total,ROBERTA_CLINICAL,NaN,todas,264,NaN
4,correctos_por_clase,ROBERTA_CLINICAL,NaN,ansiedad,75,NaN
5,correctos_por_clase,ROBERTA_CLINICAL,NaN,depresion,189,NaN
6,correctos_total,HIBRIDO_FINAL,NaN,todas,269,NaN
7,correctos_por_clase,HIBRIDO_FINAL,NaN,ansiedad,57,NaN
8,correctos_por_clase,HIBRIDO_FINAL,NaN,depresion,212,NaN
9,interseccion_correctos,TF-IDF,ROBERTA_CLINICAL,todas,226,NaN


# Comparación de aportes entre modelos

Modelos comparados: `TF-IDF`, `ROBERTA_CLINICAL` y `HIBRIDO_FINAL` sobre el mismo `dev_denoised`.

## Lectura rápida
- Mejor baseline simple fuerte: `TF-IDF` (0.7406).
- Mejor transformer standalone: `ROBERTA_CLINICAL`.
- Híbrido final congelado en dev: `B_A_llm0_sent0_beto1_tpl0_py_XGB_sin_feat_sin_medication|py|XGB`.

| tipo                   | modelo_a         | modelo_b                       | clase     |   n_casos | detalle                                            |
|:-----------------------|:-----------------|:-------------------------------|:----------|----------:|:---------------------------------------------------|
| correctos_total        | TF-IDF           |                                | todas     |       263 |                                                    |
| correctos_por_clase    | TF-IDF           |                                | ansiedad  |        77 |                                                    |
| correctos_por_clase    | TF-IDF           |                                | depresion |       186 |                                                    |
| correctos_total        | ROBERTA_CLINICAL |                                | todas     |       264 |                                                    |
| correctos_por_clase    | ROBERTA_CLINICAL |                                | ansiedad  |        75 |                                                    |
| correctos_por_clase    | ROBERTA_CLINICAL |                                | depresion |       189 |                                                    |
| correctos_total        | HIBRIDO_FINAL    |                                | todas     |       269 |                                                    |
| correctos_por_clase    | HIBRIDO_FINAL    |                                | ansiedad  |        57 |                                                    |
| correctos_por_clase    | HIBRIDO_FINAL    |                                | depresion |       212 |                                                    |
| interseccion_correctos | TF-IDF           | ROBERTA_CLINICAL               | todas     |       226 |                                                    |
| interseccion_correctos | TF-IDF           | ROBERTA_CLINICAL               | ansiedad  |        66 |                                                    |
| interseccion_correctos | TF-IDF           | ROBERTA_CLINICAL               | depresion |       160 |                                                    |
| interseccion_correctos | TF-IDF           | HIBRIDO_FINAL                  | todas     |       222 |                                                    |
| interseccion_correctos | TF-IDF           | HIBRIDO_FINAL                  | ansiedad  |        50 |                                                    |
| interseccion_correctos | TF-IDF           | HIBRIDO_FINAL                  | depresion |       172 |                                                    |
| interseccion_correctos | ROBERTA_CLINICAL | HIBRIDO_FINAL                  | todas     |       233 |                                                    |
| interseccion_correctos | ROBERTA_CLINICAL | HIBRIDO_FINAL                  | ansiedad  |        52 |                                                    |
| interseccion_correctos | ROBERTA_CLINICAL | HIBRIDO_FINAL                  | depresion |       181 |                                                    |
| interseccion_correctos | TF-IDF           | ROBERTA_CLINICAL|HIBRIDO_FINAL | todas     |       203 | Casos que aciertan los tres modelos.               |
| solo_modelo            | TF-IDF           |                                | todas     |        18 |                                                    |
| solo_modelo            | TF-IDF           |                                | ansiedad  |        10 | acude, madre, años, paciente, control, 26          |
| solo_modelo            | TF-IDF           |                                | depresion |         8 | sigue, refiere, crisis, tratamiento, mejoria, dias |
| solo_modelo            | ROBERTA_CLINICAL |                                | todas     |         8 |                                                    |
| solo_modelo            | ROBERTA_CLINICAL |                                | ansiedad  |         6 | años, hijo, 50, ansiedad, siente, regla            |
| solo_modelo            | ROBERTA_CLINICAL |                                | depresion |         2 | encuentra, estable, refiere, vez, ayuda, semana    |
| solo_modelo            | HIBRIDO_FINAL    |                                | todas     |        17 |                                                    |
| solo_modelo            | HIBRIDO_FINAL    |                                | ansiedad  |         4 | acude, control, refiere, mamá, niega, auto         |
| solo_modelo            | HIBRIDO_FINAL    |                                | depresion |        13 | crisis, refiere, consulta, momentos, niega, mejor  |

## Interpretación mínima
- Si dos modelos comparten muchos aciertos, probablemente están captando un núcleo sintomático parecido.
- Si un modelo conserva casos correctos exclusivos, aporta complementariedad clínica o textual.
- El híbrido final debe leerse aquí no solo por métrica, sino por trazabilidad y señal clínica reutilizable.


## 6. Errores del modelo para IPS

In [9]:
errores_ips = pd.read_csv(OUT_DIR / 'material_ips_errores_modelo.csv')
display(errores_ips.head(20))
display(Markdown((OUT_DIR / 'material_ips_errores_modelo.md').read_text(encoding='utf-8')))


,row_id,etiqueta_original,prediccion_modelo,split,señales_detectadas,tipo_de_error,hipotesis_clinica,pregunta_para_IPS,margen_prob,patient_id,texto
0,51,ansiedad,depresion,dev,Autolesion,ansiedad→depresion; seguimiento_administrativo...,La nota parece de control/seguimiento con baja...,¿Esta nota debería considerarse diagnóstica o ...,0.9949,408270,Misma medicacion Misma medicacion Control con ...
1,65,ansiedad,depresion,dev,Animo deprimido | Sintomas somaticos,ansiedad→depresion; frontera_ambigua; solapami...,La fenomenología psiquiátrica aparece mezclada...,¿La fenomenología principal aquí es psiquiátri...,0.1982,408270,Refiere sentirse con mucho dlor desde la seman...
2,68,ansiedad,depresion,dev,Apetito aumentado | Irritabilidad,ansiedad→depresion; seguimiento_administrativo...,La nota parece de control/seguimiento con baja...,¿Esta nota debería considerarse diagnóstica o ...,0.8481,408270,"Acude a control, refiere haber iniciado usod e..."
3,72,ansiedad,depresion,dev,Sueno / insomnio,ansiedad→depresion,El modelo priorizó señales depresivas o de enl...,¿Qué señal clínica justificaría sostener ansie...,0.8037,408270,Refiere que no logró dormir bien en la ultima ...
4,156,ansiedad,depresion,dev,Fatiga,ansiedad→depresion; seguimiento_administrativo...,La nota parece de control/seguimiento con baja...,¿Esta nota debería considerarse diagnóstica o ...,0.7322,381506,paciente acude a control paciente acude a cont...
5,159,ansiedad,depresion,dev,Baja concentracion | Fatiga,ansiedad→depresion; seguimiento_administrativo,La nota parece de control/seguimiento con baja...,¿Esta nota debería considerarse diagnóstica o ...,0.9279,381506,"mc: control mc: control -paciente acude sola, ..."
6,161,ansiedad,depresion,dev,Sintomas ansiosos generales,ansiedad→depresion; seguimiento_administrativo,La nota parece de control/seguimiento con baja...,¿Esta nota debería considerarse diagnóstica o ...,0.8441,381506,mc: crisis de ansiedad mc: crisis de ansiedad ...
7,162,ansiedad,depresion,dev,Ansiedad | Baja energia | Fatiga | Sintomas an...,ansiedad→depresion; seguimiento_administrativo...,La nota parece de control/seguimiento con baja...,¿Esta nota debería considerarse diagnóstica o ...,0.6865,381506,"mc: ansiedad, sic paciente mc: ansiedad, sic p..."
8,165,ansiedad,depresion,dev,Sintomas ansiosos generales,ansiedad→depresion; seguimiento_administrativo,La nota parece de control/seguimiento con baja...,¿Esta nota debería considerarse diagnóstica o ...,0.6950,381506,MC: CONTROL MC: CONTROL -BUENA EVOLUCION MC: C...
9,166,ansiedad,depresion,dev,Sintomas ansiosos generales,ansiedad→depresion; seguimiento_administrativo,La nota parece de control/seguimiento con baja...,¿Esta nota debería considerarse diagnóstica o ...,0.9707,381506,mc: control mc: control -paciente refiere que ...


# Errores del modelo final para discusión con IPS

La tabla completa está en `material_ips_errores_modelo.csv`.

- Total de errores analizados: 74
- Error global en dev: 0.2157

## Casos curados para revisión clínica
|   row_id | etiqueta_original   | prediccion_modelo   | tipo_de_error                                                                | señales_detectadas                                                              | hipotesis_clinica                                                                                                                                           | pregunta_para_IPS                                                                                         |
|---------:|:--------------------|:--------------------|:-----------------------------------------------------------------------------|:--------------------------------------------------------------------------------|:------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------|
|       51 | ansiedad            | depresion           | ansiedad→depresion; seguimiento_administrativo; nota_breve                   | Autolesion                                                                      | La nota parece de control/seguimiento con baja fenomenología activa; conviene revisar si la etiqueta representa el episodio actual o el antecedente global. | ¿Esta nota debería considerarse diagnóstica o corresponde más bien a seguimiento/gestión del tratamiento? |
|       68 | ansiedad            | depresion           | ansiedad→depresion; seguimiento_administrativo; solapamiento_medico          | Apetito aumentado | Irritabilidad                                               | La nota parece de control/seguimiento con baja fenomenología activa; conviene revisar si la etiqueta representa el episodio actual o el antecedente global. | ¿Esta nota debería considerarse diagnóstica o corresponde más bien a seguimiento/gestión del tratamiento? |
|      156 | ansiedad            | depresion           | ansiedad→depresion; seguimiento_administrativo; solapamiento_medico          | Fatiga                                                                          | La nota parece de control/seguimiento con baja fenomenología activa; conviene revisar si la etiqueta representa el episodio actual o el antecedente global. | ¿Esta nota debería considerarse diagnóstica o corresponde más bien a seguimiento/gestión del tratamiento? |
|      159 | ansiedad            | depresion           | ansiedad→depresion; seguimiento_administrativo                               | Baja concentracion | Fatiga                                                     | La nota parece de control/seguimiento con baja fenomenología activa; conviene revisar si la etiqueta representa el episodio actual o el antecedente global. | ¿Esta nota debería considerarse diagnóstica o corresponde más bien a seguimiento/gestión del tratamiento? |
|      161 | ansiedad            | depresion           | ansiedad→depresion; seguimiento_administrativo                               | Sintomas ansiosos generales                                                     | La nota parece de control/seguimiento con baja fenomenología activa; conviene revisar si la etiqueta representa el episodio actual o el antecedente global. | ¿Esta nota debería considerarse diagnóstica o corresponde más bien a seguimiento/gestión del tratamiento? |
|       65 | ansiedad            | depresion           | ansiedad→depresion; frontera_ambigua; solapamiento_medico                    | Animo deprimido | Sintomas somaticos                                            | La fenomenología psiquiátrica aparece mezclada con clínica médica o efectos de tratamiento, lo que reduce especificidad diagnóstica.                        | ¿La fenomenología principal aquí es psiquiátrica o está dominada por enfermedad médica/medicación?        |
|      252 | ansiedad            | depresion           | ansiedad→depresion; frontera_ambigua; seguimiento_administrativo; nota_breve | Autolesion                                                                      | La nota parece de control/seguimiento con baja fenomenología activa; conviene revisar si la etiqueta representa el episodio actual o el antecedente global. | ¿Esta nota debería considerarse diagnóstica o corresponde más bien a seguimiento/gestión del tratamiento? |
|      253 | ansiedad            | depresion           | ansiedad→depresion; frontera_ambigua; seguimiento_administrativo; nota_breve | Autolesion                                                                      | La nota parece de control/seguimiento con baja fenomenología activa; conviene revisar si la etiqueta representa el episodio actual o el antecedente global. | ¿Esta nota debería considerarse diagnóstica o corresponde más bien a seguimiento/gestión del tratamiento? |
|      273 | ansiedad            | depresion           | ansiedad→depresion; frontera_ambigua; solapamiento_medico                    | Animo deprimido | Retraimiento social / aislamiento                             | La fenomenología psiquiátrica aparece mezclada con clínica médica o efectos de tratamiento, lo que reduce especificidad diagnóstica.                        | ¿La fenomenología principal aquí es psiquiátrica o está dominada por enfermedad médica/medicación?        |
|      520 | ansiedad            | depresion           | ansiedad→depresion; frontera_ambigua; seguimiento_administrativo; nota_breve | Autolesion                                                                      | La nota parece de control/seguimiento con baja fenomenología activa; conviene revisar si la etiqueta representa el episodio actual o el antecedente global. | ¿Esta nota debería considerarse diagnóstica o corresponde más bien a seguimiento/gestión del tratamiento? |
|      162 | ansiedad            | depresion           | ansiedad→depresion; seguimiento_administrativo; solapamiento_sintomatico     | Ansiedad | Baja energia | Fatiga | Sintomas ansiosos generales | Sueno alterado | La nota parece de control/seguimiento con baja fenomenología activa; conviene revisar si la etiqueta representa el episodio actual o el antecedente global. | ¿Esta nota debería considerarse diagnóstica o corresponde más bien a seguimiento/gestión del tratamiento? |


## 7. Preguntas estructuradas para IPS

In [10]:
display(Markdown((OUT_DIR / 'preguntas_sugeridas_ips.md').read_text(encoding='utf-8')))


# Preguntas sugeridas para IPS

## Sobre señales clínicas
- ¿Los patrones más recurrentes asociados a ansiedad y depresión tienen sentido clínico en estas notas de Paraguay?
- ¿Hay señales locales, coloquialismos o abreviaturas clínicas que todavía estemos subcapturando?

## Sobre errores y etiquetas
- ¿Los errores ansiedad→depresion y depresión→ansiedad corresponden a error real del modelo o a consultas clínicamente ambiguas?
- ¿Hay consultas de seguimiento donde la etiqueta global del paciente no coincide con la fenomenología de esa consulta puntual?
- ¿Qué tipo de notas deberían considerarse poco diagnósticas o no apropiadas para esta tarea diferencial?

## Sobre filtrado
- ¿Los casos removidos como reposición/control sin señal clínica útil están bien excluidos o conviene rescatar alguno?
- ¿La distinción entre negación del paciente y negación de plantilla/médico coincide con la práctica clínica?

## Sobre futuros datos
- ¿Sería viable obtener un grupo de control o más consultas con baja carga psiquiátrica para una fase posterior?
- ¿Qué subconjunto de casos convendría reetiquetar primero si se decide una validación experta consulta por consulta?


## 8. Justificación metodológica y apoyo de literatura

## 8b. Trazabilidad de scripts backend

Este notebook es una capa de lectura. La generación reproducible de artefactos se delega a scripts para evitar curación manual dentro del notebook:

- `scripts/export/generar_material_validacion_ips.py`: arma el paquete base de validación clínica.
- `scripts/export/curar_dossier_ips.py`: genera el dossier clínico curado para IPS, tesis y futura xAI.
- `scripts/export/cerrar_fase_ips.py`: consolida auditoría del dataset, contraste crudo vs filtrado, glosario y guía de defensa.

Esta separación es la opción adecuada en esta fase porque mejora trazabilidad y evita que el notebook quede como única fuente de verdad.

In [11]:
display(Markdown((OUT_DIR / 'justificacion_metodologica_y_clinica.md').read_text(encoding='utf-8')))
display(Markdown((OUT_DIR / 'preguntas_notebooklm_validacion_clinica.md').read_text(encoding='utf-8')))


# Justificación metodológica y clínica

- Mostrar el preprocesamiento y el filtrado importa porque el rendimiento depende del universo de notas realmente modelado, no solo del clasificador.
- En un problema diferencial entre ansiedad y depresión, el desbalance afecta la lectura de las métricas; por eso conviene priorizar `macro_f1`, `balanced_accuracy` y F1 por clase.
- Validar errores con psiquiatras importa porque parte del desacuerdo modelo-etiqueta puede reflejar baja separabilidad clínica de la consulta y no solo una falla algorítmica.
- En esta fase sigue siendo razonable trabajar sin grupo de control porque la tarea actual es diferencial dentro de una población ya clínica; eso no elimina el valor de un grupo de control como extensión futura.
- La validación clínica antes de abrir `test` ayuda a cerrar supuestos sobre señales relevantes, notas poco diagnósticas y posibles límites del etiquetado actual.


# Preguntas sugeridas para NotebookLM sobre validación clínica

- ¿Qué literatura clínica o de cNLP justifica revisar errores de clasificación con expertos antes de una evaluación final en hold-out?
- ¿Qué evidencia existe sobre label noise o baja especificidad diagnóstica en notas de seguimiento dentro de EHR psiquiátricos?
- ¿Cómo se discute en la literatura la diferencia entre etiqueta a nivel paciente y etiqueta a nivel consulta en tareas de fenotipado clínico?
- ¿Qué argumentos metodológicos sostienen usar `macro_f1` y `balanced_accuracy` en tareas binarias desbalanceadas en salud mental?
- ¿Qué se reporta sobre la utilidad y los límites de trabajar sin grupo de control en tareas diferenciales acotadas dentro de población clínica?
- ¿Qué enfoques se usan para validar con expertos señales clínicas detectadas por modelos híbridos o reglas en EHR?
- ¿Qué trabajos discuten cómo presentar ejemplos de error, ambigüedad clínica y consultas poco diagnósticas en tesis o papers de NLP clínico?
